<a href="https://colab.research.google.com/github/PA-Pal601/E-Commerce/blob/main/Emotion_Recognition_from_Speech.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Emotion Recognition from Speech

## Architecture Overview

```
Raw Audio → wav2vec2 / HuBERT (SSL Pre-training)
         → Fine-tuned Transformer Encoder
         → Multi-Head Attention Pooling
         → Emotion Classifier
```


### Datasets Used
- **RAVDESS** — 24 actors, 8 emotions, 7356 files
- **CREMA-D** — 7442 clips, 91 actors, 6 emotions
- **TESS** — 2800 files, 2 actors, 7 emotions

**Target Emotions:** Neutral · Calm · Happy · Sad · Angry · Fearful · Disgust · Surprised

## Install Dependencies

In [1]:
%%capture
!pip install transformers datasets librosa soundfile audiomentations torch torchvision torchaudio
!pip install accelerate evaluate scikit-learn matplotlib seaborn tqdm pandas
!pip install kaggle einops timm
!pip install pyroomacoustics
print('✅ All packages installed!')

## Imports & Global Config

In [2]:
import os, random, warnings, math, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from collections import Counter
from tqdm.notebook import tqdm
import IPython.display as ipd
warnings.filterwarnings('ignore')

import librosa
import librosa.display
import soundfile as sf

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR, CosineAnnealingWarmRestarts
from torch.cuda.amp import GradScaler, autocast

from transformers import (
    Wav2Vec2Model, Wav2Vec2Processor,
    HubertModel, AutoFeatureExtractor
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score
)
from sklearn.preprocessing import LabelEncoder
import audiomentations as AA

# ─── Reproducibility ─────────────────────────────────────────────────────────
SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = True
seed_everything()

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

Device: cpu


In [3]:
# ─── Global Hyperparameters ───────────────────────────────────────────────────
class CFG:
    # Audio
    SR           = 16000        # wav2vec2 expects 16kHz
    DURATION     = 4            # seconds
    N_SAMPLES    = SR * DURATION

    # Model
    BACKBONE     = 'facebook/wav2vec2-base'   # swap for 'facebook/hubert-base-ls960'
    FREEZE_FEAT  = True          # freeze feature extractor
    UNFREEZE_LAYERS = 4          # fine-tune last N transformer layers
    HIDDEN_DIM   = 256
    N_HEADS_POOL = 8             # attention pooling heads
    DROPOUT      = 0.3

    # Training
    EPOCHS       = 30
    BATCH_SIZE   = 16
    LR           = 2e-4
    WEIGHT_DECAY = 1e-4
    LABEL_SMOOTH = 0.1
    FOCAL_GAMMA  = 2.0
    MIXUP_ALPHA  = 0.4
    N_FOLDS      = 5
    USE_AMP      = True          # mixed precision

    # Augmentation
    USE_SPECAUGMENT = True
    USE_MIXUP    = True

    # Emotions
    EMOTIONS     = ['neutral','calm','happy','sad','angry','fearful','disgust','surprised']
    N_CLASSES    = len(EMOTIONS)

    EMOTION_COLORS = {
        'neutral':'#95a5a6','calm':'#3498db','happy':'#f1c40f',
        'sad':'#2980b9','angry':'#e74c3c','fearful':'#9b59b6',
        'disgust':'#27ae60','surprised':'#e67e22'
    }

print(f'Config loaded — {CFG.N_CLASSES} emotion classes')
print(f'Backbone: {CFG.BACKBONE}')

Config loaded — 8 emotion classes
Backbone: facebook/wav2vec2-base


###📥 Dataset Download (RAVDESS + CREMA-D + TESS)

> Synthetic dataset.

In [4]:
# Synthetic dataset (instant, no download) ───────────────────────
print('Using synthetic audio — replace with real RAVDESS/CREMA-D/TESS for full accuracy.')

def make_synthetic_audio(emotion, sr=CFG.SR, duration=CFG.DURATION, seed=0):
    np.random.seed(seed)
    t = np.linspace(0, duration, sr * duration)
    params = {
        'neutral':   dict(f0=180, jitter=0.01,  energy=0.3),
        'calm':      dict(f0=160, jitter=0.005, energy=0.2),
        'happy':     dict(f0=240, jitter=0.02,  energy=0.7),
        'sad':       dict(f0=130, jitter=0.008, energy=0.2),
        'angry':     dict(f0=280, jitter=0.04,  energy=0.9),
        'fearful':   dict(f0=220, jitter=0.03,  energy=0.5),
        'disgust':   dict(f0=170, jitter=0.025, energy=0.4),
        'surprised': dict(f0=300, jitter=0.035, energy=0.8),
    }[emotion]
    f0  = params['f0'] * (1 + params['jitter'] * np.random.randn(len(t)))
    sig = params['energy'] * np.sin(2 * np.pi * np.cumsum(f0) / sr)
    for h in [2, 3, 4]:
        sig += (params['energy'] / (h * 2)) * np.sin(2 * np.pi * h * np.cumsum(f0) / sr)
    sig += 0.015 * np.random.randn(len(t))
    return (sig / (np.max(np.abs(sig)) + 1e-9)).astype(np.float32)

# Build synthetic dataframe
N_PER_CLASS = 100
records = []
for emo in CFG.EMOTIONS:
    for i in range(N_PER_CLASS):
        records.append({'emotion': emo, 'seed': i * 10 + CFG.EMOTIONS.index(emo), 'source': 'synthetic'})
df = pd.DataFrame(records)
df['label'] = df['emotion'].map({e:i for i,e in enumerate(CFG.EMOTIONS)})
print(f'Dataset: {len(df)} samples across {CFG.N_CLASSES} classes')
print(df['emotion'].value_counts().to_string())

Using synthetic audio — replace with real RAVDESS/CREMA-D/TESS for full accuracy.
Dataset: 800 samples across 8 classes
emotion
neutral      100
calm         100
happy        100
sad          100
angry        100
fearful      100
disgust      100
surprised    100


## 🔊 Advanced Audio Augmentation Pipeline

In [5]:
# ─── audiomentations augmentation pipeline ────────────────────────────────────
train_augment = AA.Compose([
    AA.AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.4),
    AA.TimeStretch(min_rate=0.85, max_rate=1.15, p=0.4),
    AA.PitchShift(min_semitones=-3, max_semitones=3, p=0.4),
    AA.Shift(min_shift=-0.2, max_shift=0.2, p=0.3),
    AA.RoomSimulator(p=0.2),
    AA.Gain(min_gain_db=-6, max_gain_db=6, p=0.3),
    AA.LowPassFilter(min_cutoff_freq=2000, max_cutoff_freq=7500, p=0.2),
])

# ─── SpecAugment (applied on mel-spectrogram / features) ─────────────────────
class SpecAugment(nn.Module):
    """
    SpecAugment: time masking + frequency masking on hidden states.
    Park et al., 2019 — https://arxiv.org/abs/1904.08779
    """
    def __init__(self, freq_mask=27, time_mask=70, n_freq_masks=2, n_time_masks=2):
        super().__init__()
        self.freq_mask   = freq_mask
        self.time_mask   = time_mask
        self.n_freq      = n_freq_masks
        self.n_time      = n_time_masks

    def forward(self, x):   # x: (B, T, F)
        if not self.training:
            return x
        B, T, F = x.shape
        for _ in range(self.n_freq):
            f = random.randint(0, self.freq_mask)
            f0 = random.randint(0, max(F - f, 1))
            x[:, :, f0:f0 + f] = 0
        for _ in range(self.n_time):
            t = random.randint(0, min(self.time_mask, T - 1))
            t0 = random.randint(0, max(T - t, 1))
            x[:, t0:t0 + t, :] = 0
        return x

print('✅ Augmentation pipeline ready')

✅ Augmentation pipeline ready


## 🗃️ Dataset & DataLoader

In [6]:
class SpeechEmotionDataset(Dataset):
    """
    Loads raw waveforms and feeds them to wav2vec2 processor.
    Supports both file-path mode and synthetic mode.
    """
    def __init__(self, df, processor, augment=None, is_train=True):
        self.df        = df.reset_index(drop=True)
        self.processor = processor
        self.augment   = augment
        self.is_train  = is_train

    def __len__(self):
        return len(self.df)

    def _load_audio(self, row):
        if 'path' in row and pd.notna(row.get('path', None)):
            audio, _ = librosa.load(row['path'], sr=CFG.SR, duration=CFG.DURATION)
        else:
            audio = make_synthetic_audio(row['emotion'], seed=int(row['seed']))
        # Pad / trim to fixed length
        if len(audio) < CFG.N_SAMPLES:
            audio = np.pad(audio, (0, CFG.N_SAMPLES - len(audio)))
        else:
            audio = audio[:CFG.N_SAMPLES]
        return audio.astype(np.float32)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        audio = self._load_audio(row)

        if self.is_train and self.augment is not None:
            audio = self.augment(samples=audio, sample_rate=CFG.SR)

        inputs = self.processor(
            audio,
            sampling_rate=CFG.SR,
            return_tensors='pt',
            padding='max_length',
            max_length=CFG.N_SAMPLES,
            truncation=True
        )
        input_values = inputs.input_values.squeeze(0)  # (N_SAMPLES,)
        label        = torch.tensor(int(row['label']), dtype=torch.long)
        return input_values, label


def make_weighted_sampler(labels):
    """Inverse-frequency weighted sampler to handle class imbalance."""
    counts  = Counter(labels)
    weights = [1.0 / counts[l] for l in labels]
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

print('✅ Dataset class defined')

✅ Dataset class defined


## 🏗️ SOTA Model Architecture

### wav2vec2 + Conformer + Multi-Head Attention Pooling

In [7]:
# ─── Multi-Head Attention Pooling ─────────────────────────────────────────────
class AttentionPooling(nn.Module):
    """
    Learns a weighted sum over time frames.
    Outperforms simple mean/max pooling on speech tasks.
    Ref: Safari & Hernandez-Lobato, 2020.
    """
    def __init__(self, hidden_dim, n_heads=8):
        super().__init__()
        self.n_heads  = n_heads
        self.attn     = nn.Linear(hidden_dim, n_heads)
        self.proj     = nn.Linear(hidden_dim * n_heads, hidden_dim)
        self.norm     = nn.LayerNorm(hidden_dim)

    def forward(self, x, mask=None):   # x: (B, T, D)
        scores = self.attn(x)          # (B, T, H)
        if mask is not None:
            scores = scores.masked_fill(mask.unsqueeze(-1), float('-inf'))
        weights = torch.softmax(scores, dim=1)  # (B, T, H)
        # weighted sum per head
        heads = []
        for h in range(self.n_heads):
            heads.append((x * weights[:, :, h:h+1]).sum(dim=1))   # (B, D)
        out = torch.cat(heads, dim=-1)   # (B, D*H)
        out = self.proj(out)              # (B, D)
        return self.norm(out)


# ─── Conformer Block ──────────────────────────────────────────────────────────
class ConformerBlock(nn.Module):
    """
    Conformer = FF + MHSA + Conv + FF (Gulati et al., 2020).
    Captures both local (conv) and global (attention) dependencies.
    """
    def __init__(self, dim, n_heads=4, conv_kernel=31, dropout=0.1):
        super().__init__()
        # Feed-forward 1
        self.ff1   = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim * 4), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(dim * 4, dim), nn.Dropout(dropout)
        )
        # Multi-head self-attention
        self.norm_attn = nn.LayerNorm(dim)
        self.mhsa      = nn.MultiheadAttention(dim, n_heads, dropout=dropout, batch_first=True)
        self.drop_attn = nn.Dropout(dropout)
        # Conv module
        self.norm_conv = nn.LayerNorm(dim)
        self.conv = nn.Sequential(
            nn.Conv1d(dim, 2 * dim, 1),
            nn.GLU(dim=1),
            nn.Conv1d(dim, dim, conv_kernel, padding=conv_kernel // 2, groups=dim),
            nn.BatchNorm1d(dim),
            nn.SiLU(),
            nn.Conv1d(dim, dim, 1),
            nn.Dropout(dropout)
        )
        # Feed-forward 2
        self.ff2   = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim * 4), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(dim * 4, dim), nn.Dropout(dropout)
        )
        self.final_norm = nn.LayerNorm(dim)

    def forward(self, x):          # x: (B, T, D)
        x = x + 0.5 * self.ff1(x)
        r = self.norm_attn(x)
        r, _ = self.mhsa(r, r, r)
        x = x + self.drop_attn(r)
        r = self.norm_conv(x).transpose(1, 2)   # (B, D, T)
        r = self.conv(r).transpose(1, 2)         # (B, T, D)
        x = x + r
        x = x + 0.5 * self.ff2(x)
        return self.final_norm(x)


print('✅ Conformer block & Attention Pooling defined')

✅ Conformer block & Attention Pooling defined


In [8]:
# ─── Full SOTA SER Model ───────────────────────────────────────────────────────
class SOTASpeechEmotionModel(nn.Module):
    """
    wav2vec2 (frozen feature extractor) → fine-tuned transformer layers
    → SpecAugment → 2× Conformer blocks → Multi-head Attention Pooling
    → Classifier head with label smoothing + focal loss
    """
    def __init__(self, cfg):
        super().__init__()
        # ── Backbone ──────────────────────────────────────────────────────────
        self.backbone = Wav2Vec2Model.from_pretrained(cfg.BACKBONE)
        backbone_dim  = self.backbone.config.hidden_size   # 768 for base

        # Freeze feature extractor CNN
        if cfg.FREEZE_FEAT:
            for p in self.backbone.feature_extractor.parameters():
                p.requires_grad = False
            for p in self.backbone.feature_projection.parameters():
                p.requires_grad = False

        # Only fine-tune last N transformer layers
        n_layers = len(self.backbone.encoder.layers)
        for i, layer in enumerate(self.backbone.encoder.layers):
            if i < n_layers - cfg.UNFREEZE_LAYERS:
                for p in layer.parameters():
                    p.requires_grad = False

        # ── SpecAugment ───────────────────────────────────────────────────────
        self.spec_aug = SpecAugment(freq_mask=27, time_mask=70)

        # ── Projection ────────────────────────────────────────────────────────
        self.proj = nn.Sequential(
            nn.Linear(backbone_dim, cfg.HIDDEN_DIM),
            nn.LayerNorm(cfg.HIDDEN_DIM),
            nn.GELU(),
            nn.Dropout(cfg.DROPOUT)
        )

        # ── Conformer layers ──────────────────────────────────────────────────
        self.conformer = nn.Sequential(*[
            ConformerBlock(cfg.HIDDEN_DIM, n_heads=4, dropout=cfg.DROPOUT)
            for _ in range(2)
        ])

        # ── Attention Pooling ─────────────────────────────────────────────────
        self.pool = AttentionPooling(cfg.HIDDEN_DIM, n_heads=cfg.N_HEADS_POOL)

        # ── Classifier head ───────────────────────────────────────────────────
        self.classifier = nn.Sequential(
            nn.Linear(cfg.HIDDEN_DIM, cfg.HIDDEN_DIM // 2),
            nn.GELU(),
            nn.Dropout(cfg.DROPOUT),
            nn.Linear(cfg.HIDDEN_DIM // 2, cfg.N_CLASSES)
        )

    def forward(self, input_values, attention_mask=None):
        out    = self.backbone(input_values, attention_mask=attention_mask)
        hidden = out.last_hidden_state   # (B, T, 768)

        if CFG.USE_SPECAUGMENT:
            hidden = self.spec_aug(hidden)

        hidden = self.proj(hidden)       # (B, T, HIDDEN_DIM)
        hidden = self.conformer(hidden)  # (B, T, HIDDEN_DIM)
        pooled = self.pool(hidden)       # (B, HIDDEN_DIM)
        logits = self.classifier(pooled) # (B, N_CLASSES)
        return logits


# Quick parameter count
model_test = SOTASpeechEmotionModel(CFG).to(DEVICE)
total   = sum(p.numel() for p in model_test.parameters())
trainable = sum(p.numel() for p in model_test.parameters() if p.requires_grad)
print(f'\nModel: {total/1e6:.1f}M total params | {trainable/1e6:.1f}M trainable')
del model_test; torch.cuda.empty_cache()

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.bias             | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Model: 98.2M total params | 36.9M trainable


## 📉 Advanced Loss Functions

In [9]:
# ─── Focal Loss with Label Smoothing ─────────────────────────────────────────
class FocalLoss(nn.Module):
    """
    Focal Loss (Lin et al., 2017) + Label Smoothing.
    Down-weights easy examples, focuses on hard ones.
    Better than CE for imbalanced emotion classes.
    """
    def __init__(self, gamma=2.0, smoothing=0.1, weight=None):
        super().__init__()
        self.gamma     = gamma
        self.smoothing = smoothing
        self.weight    = weight

    def forward(self, logits, targets):
        n_cls = logits.size(-1)
        # Label smoothing
        with torch.no_grad():
            smooth_targets = torch.full_like(logits, self.smoothing / (n_cls - 1))
            smooth_targets.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
        log_probs = F.log_softmax(logits, dim=-1)
        probs     = torch.exp(log_probs)
        focal_w   = (1 - probs) ** self.gamma
        loss      = -(focal_w * smooth_targets * log_probs).sum(dim=-1)
        return loss.mean()


# ─── Mixup Augmentation ───────────────────────────────────────────────────────
def mixup_batch(x, y, alpha=0.4):
    """
    Mixup: linearly interpolate pairs of examples.
    Zhang et al., 2018 — https://arxiv.org/abs/1710.09412
    Returns mixed inputs and a (lam, y_a, y_b) tuple for loss computation.
    """
    lam   = np.random.beta(alpha, alpha)
    B     = x.size(0)
    idx   = torch.randperm(B, device=x.device)
    mixed = lam * x + (1 - lam) * x[idx]
    return mixed, lam, y, y[idx]


def mixup_loss(criterion, logits, lam, y_a, y_b):
    return lam * criterion(logits, y_a) + (1 - lam) * criterion(logits, y_b)


criterion = FocalLoss(gamma=CFG.FOCAL_GAMMA, smoothing=CFG.LABEL_SMOOTH)
print('✅ FocalLoss + Label Smoothing + Mixup ready')

✅ FocalLoss + Label Smoothing + Mixup ready


##  Training Engine with Mixed Precision + Cosine Warmup

In [10]:
def get_optimizer_and_scheduler(model, n_steps):
    """
    Differential LR: lower LR for backbone, higher for task head.
    OneCycleLR with cosine annealing + linear warmup.
    """
    backbone_params = [p for n, p in model.named_parameters()
                       if 'backbone' in n and p.requires_grad]
    head_params     = [p for n, p in model.named_parameters()
                       if 'backbone' not in n and p.requires_grad]

    optimizer = AdamW([
        {'params': backbone_params, 'lr': CFG.LR * 0.1},
        {'params': head_params,     'lr': CFG.LR}
    ], weight_decay=CFG.WEIGHT_DECAY)

    scheduler = OneCycleLR(
        optimizer,
        max_lr=[CFG.LR * 0.1, CFG.LR],
        total_steps=n_steps,
        pct_start=0.1,
        anneal_strategy='cos',
        div_factor=25,
        final_div_factor=1000
    )
    return optimizer, scheduler


class EarlyStopping:
    def __init__(self, patience=7, delta=0.001):
        self.patience  = patience
        self.delta     = delta
        self.counter   = 0
        self.best_score = None
        self.best_weights = None

    def __call__(self, score, model):
        if self.best_score is None or score > self.best_score + self.delta:
            self.best_score   = score
            self.best_weights = copy.deepcopy(model.state_dict())
            self.counter      = 0
            return False   # don't stop
        self.counter += 1
        return self.counter >= self.patience


def train_one_epoch(model, loader, optimizer, scheduler, scaler, criterion, epoch):
    model.train()
    total_loss, total_correct, total_n = 0., 0, 0
    pbar = tqdm(loader, desc=f'Epoch {epoch:02d} [Train]', leave=False)

    for x, y in pbar:
        x, y = x.to(DEVICE), y.to(DEVICE)

        # Mixup
        if CFG.USE_MIXUP and random.random() < 0.5:
            x_mix, lam, y_a, y_b = mixup_batch(x, y, CFG.MIXUP_ALPHA)
            with autocast(enabled=CFG.USE_AMP):
                logits = model(x_mix)
                loss   = mixup_loss(criterion, logits, lam, y_a, y_b)
        else:
            with autocast(enabled=CFG.USE_AMP):
                logits = model(x)
                loss   = criterion(logits, y)

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        preds = logits.argmax(-1)
        total_correct += (preds == y).sum().item()
        total_loss    += loss.item() * x.size(0)
        total_n       += x.size(0)
        pbar.set_postfix(loss=f'{total_loss/total_n:.4f}', acc=f'{total_correct/total_n:.3f}')

    return total_loss / total_n, total_correct / total_n


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total_loss, all_preds, all_labels = 0., [], []

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with autocast(enabled=CFG.USE_AMP):
            logits = model(x)
            loss   = criterion(logits, y)
        total_loss += loss.item() * x.size(0)
        all_preds.extend(logits.argmax(-1).cpu().numpy())
        all_labels.extend(y.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='weighted')
    return total_loss / len(loader.dataset), acc, f1, np.array(all_preds), np.array(all_labels)


print('✅ Training utilities ready')

✅ Training utilities ready


## Stratified K-Fold Cross Validation Training

In [11]:
# ─── Load processor once ──────────────────────────────────────────────────────
print(f'Loading processor: {CFG.BACKBONE}')
processor = Wav2Vec2Processor.from_pretrained(CFG.BACKBONE)
print('✅ Processor loaded')

Loading processor: facebook/wav2vec2-base


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

✅ Processor loaded


In [ ]:
# ─── K-Fold Training Loop ─────────────────────────────────────────────────────
skf        = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=SEED)
fold_results = []
all_fold_histories = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(df, df['label'])):
    print(f'\n{"="*60}')
    print(f'  FOLD {fold+1}/{CFG.N_FOLDS}')
    print(f'{"="*60}')

    df_tr = df.iloc[tr_idx]
    df_va = df.iloc[va_idx]

    tr_ds = SpeechEmotionDataset(df_tr, processor, augment=train_augment, is_train=True)
    va_ds = SpeechEmotionDataset(df_va, processor, augment=None,          is_train=False)

    sampler  = make_weighted_sampler(df_tr['label'].tolist())
    tr_loader = DataLoader(tr_ds, batch_size=CFG.BATCH_SIZE, sampler=sampler,
                           num_workers=2, pin_memory=True)
    va_loader = DataLoader(va_ds, batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
                           num_workers=2, pin_memory=True)

    model     = SOTASpeechEmotionModel(CFG).to(DEVICE)
    n_steps   = CFG.EPOCHS * len(tr_loader)
    optimizer, scheduler = get_optimizer_and_scheduler(model, n_steps)
    scaler    = GradScaler(enabled=CFG.USE_AMP)
    es        = EarlyStopping(patience=7)
    history   = {'train_loss':[],'train_acc':[],'val_loss':[],'val_acc':[],'val_f1':[]}

    for epoch in range(1, CFG.EPOCHS + 1):
        tr_loss, tr_acc = train_one_epoch(model, tr_loader, optimizer, scheduler,
                                          scaler, criterion, epoch)
        va_loss, va_acc, va_f1, _, _ = validate(model, va_loader, criterion)

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(va_loss)
        history['val_acc'].append(va_acc)
        history['val_f1'].append(va_f1)

        print(f'  Ep {epoch:02d} | tr_loss={tr_loss:.4f} tr_acc={tr_acc:.3f} '
              f'| val_loss={va_loss:.4f} val_acc={va_acc:.3f} val_f1={va_f1:.3f}')

        if es(va_f1, model):
            print(f'  ⏹  Early stopping at epoch {epoch}')
            break

    model.load_state_dict(es.best_weights)
    _, best_acc, best_f1, preds, labels = validate(model, va_loader, criterion)
    fold_results.append({'fold': fold+1, 'acc': best_acc, 'f1': best_f1,
                         'preds': preds, 'labels': labels})
    all_fold_histories.append(history)
    torch.save(model.state_dict(), f'ser_model_fold{fold+1}.pt')
    print(f'\n  ✅ Fold {fold+1} | Best Acc: {best_acc*100:.2f}% | Best WF1: {best_f1:.4f}')

print(f'\n{"="*60}')
mean_acc = np.mean([r["acc"] for r in fold_results])
mean_f1  = np.mean([r["f1"]  for r in fold_results])
print(f'CV RESULTS: Acc={mean_acc*100:.2f}% ± {np.std([r["acc"] for r in fold_results])*100:.2f}%')
print(f'            WF1={mean_f1:.4f} ± {np.std([r["f1"] for r in fold_results]):.4f}')


  FOLD 1/5


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.bias             | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 01 [Train]:   0%|          | 0/40 [00:00<?, ?it/s]

## 📊 Evaluation & Visualisation

In [ ]:
# ─── Training curves (all folds) ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
colors = plt.cm.tab10(np.linspace(0, 1, CFG.N_FOLDS))

for i, (h, r) in enumerate(zip(all_fold_histories, fold_results)):
    axes[0].plot(h['val_acc'],  label=f'Fold {i+1} ({r["acc"]*100:.1f}%)', color=colors[i], linewidth=2)
    axes[1].plot(h['val_f1'],   label=f'Fold {i+1} ({r["f1"]:.3f})',       color=colors[i], linewidth=2)

for ax, title, ylabel in zip(axes,
    ['Validation Accuracy per Fold', 'Validation Weighted-F1 per Fold'],
    ['Accuracy', 'Weighted F1']):
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle('K-Fold Cross-Validation Training Curves', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── Aggregate confusion matrix across folds ─────────────────────────────────
all_preds  = np.concatenate([r['preds']  for r in fold_results])
all_labels = np.concatenate([r['labels'] for r in fold_results])

cm   = confusion_matrix(all_labels, all_preds)
cm_n = cm.astype(float) / cm.sum(axis=1, keepdims=True)   # normalised

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.heatmap(cm,   annot=True, fmt='d',    cmap='Blues',  ax=axes[0],
            xticklabels=CFG.EMOTIONS, yticklabels=CFG.EMOTIONS)
axes[0].set_title('Confusion Matrix (counts)', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
axes[0].tick_params(axis='x', rotation=45)

sns.heatmap(cm_n, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[1],
            xticklabels=CFG.EMOTIONS, yticklabels=CFG.EMOTIONS)
axes[1].set_title('Confusion Matrix (normalised)', fontweight='bold')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('Aggregated Cross-Validation Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nClassification Report (aggregated CV):')
print(classification_report(all_labels, all_preds, target_names=CFG.EMOTIONS, digits=4))

In [ ]:
# ─── Per-emotion F1 radar chart ───────────────────────────────────────────────
from sklearn.metrics import f1_score
per_class_f1 = f1_score(all_labels, all_preds, average=None)

angles  = np.linspace(0, 2*np.pi, CFG.N_CLASSES, endpoint=False).tolist()
values  = per_class_f1.tolist()
angles += angles[:1]; values += values[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
ax.plot(angles, values, 'o-', linewidth=2, color='#3498db')
ax.fill(angles, values, alpha=0.25, color='#3498db')
ax.set_xticks(angles[:-1])
ax.set_xticklabels(CFG.EMOTIONS, fontsize=11)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2','0.4','0.6','0.8','1.0'], fontsize=8)
ax.set_title('Per-Emotion F1 Score (Radar)', fontsize=14, fontweight='bold', pad=20)
ax.grid(True, alpha=0.4)

for i, (angle, val, emo) in enumerate(zip(angles[:-1], values[:-1], CFG.EMOTIONS)):
    ax.annotate(f'{val:.2f}', xy=(angle, val), fontsize=8,
                ha='center', va='center',
                color=CFG.EMOTION_COLORS[emo], fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── Fold-wise accuracy bar chart ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
folds   = [r['fold'] for r in fold_results]
accs    = [r['acc'] * 100 for r in fold_results]
f1s     = [r['f1'] * 100  for r in fold_results]

x = np.arange(len(folds))
b1 = ax.bar(x - 0.2, accs, 0.38, label='Accuracy (%)', color='#3498db', edgecolor='black', linewidth=0.5)
b2 = ax.bar(x + 0.2, f1s,  0.38, label='Weighted F1 (%)', color='#e74c3c', edgecolor='black', linewidth=0.5)
ax.bar_label(b1, fmt='%.1f', fontsize=9, padding=2)
ax.bar_label(b2, fmt='%.1f', fontsize=9, padding=2)
ax.axhline(y=np.mean(accs), color='#3498db', linestyle='--', linewidth=1.5,
           label=f'Mean Acc {np.mean(accs):.1f}%')
ax.axhline(y=np.mean(f1s),  color='#e74c3c', linestyle='--', linewidth=1.5,
           label=f'Mean F1  {np.mean(f1s):.1f}%')
ax.set_xticks(x); ax.set_xticklabels([f'Fold {f}' for f in folds])
ax.set_title('K-Fold Results Summary', fontsize=14, fontweight='bold')
ax.set_ylabel('Score (%)'); ax.set_ylim(0, 115)
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Test-Time Augmentation (TTA) Inference

In [ ]:
@torch.no_grad()
def tta_predict(model, audio, processor, n_tta=5):
    """
    Test-Time Augmentation: run inference N times with slight perturbations,
    then average softmax probabilities → more robust predictions.
    """
    model.eval()
    tta_augment = AA.Compose([
        AA.AddGaussianNoise(min_amplitude=0.0005, max_amplitude=0.005, p=0.5),
        AA.Gain(min_gain_db=-3, max_gain_db=3, p=0.5),
        AA.Shift(min_shift=-0.05, max_shift=0.05, p=0.5),
    ])

    probs_sum = np.zeros(CFG.N_CLASSES)

    # Original
    inp   = processor(audio, sampling_rate=CFG.SR, return_tensors='pt',
                      padding='max_length', max_length=CFG.N_SAMPLES, truncation=True)
    logits = model(inp.input_values.squeeze(0).unsqueeze(0).to(DEVICE))
    probs_sum += F.softmax(logits, dim=-1).cpu().numpy()[0]

    # Augmented runs
    for _ in range(n_tta - 1):
        aug_audio = tta_augment(samples=audio.copy(), sample_rate=CFG.SR)
        inp       = processor(aug_audio, sampling_rate=CFG.SR, return_tensors='pt',
                              padding='max_length', max_length=CFG.N_SAMPLES, truncation=True)
        logits    = model(inp.input_values.squeeze(0).unsqueeze(0).to(DEVICE))
        probs_sum += F.softmax(logits, dim=-1).cpu().numpy()[0]

    avg_probs  = probs_sum / n_tta
    pred_idx   = np.argmax(avg_probs)
    return CFG.EMOTIONS[pred_idx], avg_probs


def visualize_prediction(emotion, probs, title='Emotion Prediction'):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Bar chart
    colors = [CFG.EMOTION_COLORS[e] for e in CFG.EMOTIONS]
    bars = ax1.barh(CFG.EMOTIONS, probs * 100, color=colors,
                    edgecolor='black', linewidth=0.4)
    ax1.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
    ax1.axvline(x=100/CFG.N_CLASSES, color='gray', linestyle='--', linewidth=1, label='Chance')
    ax1.set_xlim(0, 115)
    ax1.set_title(f'Prediction: {emotion.upper()} ({probs.max()*100:.1f}%)',
                  fontweight='bold', color=CFG.EMOTION_COLORS[emotion], fontsize=13)
    ax1.set_xlabel('Probability (%)')
    ax1.legend()

    # Pie chart
    wedge_props = dict(width=0.5, edgecolor='white')
    ax2.pie(probs, labels=CFG.EMOTIONS, colors=colors, autopct='%1.1f%%',
            startangle=140, textprops={'fontsize': 9}, wedgeprops=wedge_props)
    ax2.set_title('Probability Distribution', fontweight='bold')

    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

print('✅ TTA inference engine ready')

In [ ]:
# ─── Demo TTA predictions ─────────────────────────────────────────────────────
# Load best fold model
best_fold  = max(fold_results, key=lambda r: r['f1'])
best_model = SOTASpeechEmotionModel(CFG).to(DEVICE)
best_model.load_state_dict(torch.load(f'ser_model_fold{best_fold["fold"]}.pt', map_location=DEVICE))
print(f'Using Fold {best_fold["fold"]} model (F1={best_fold["f1"]:.4f})')

test_emotions = ['happy', 'angry', 'sad', 'fearful']
print('\n🎤  TTA Predictions (5 augmented passes each)\n')
print(f'{"True":<12}  {"Predicted":<12}  Confidence  Correct?')
print('-' * 50)

for emo in test_emotions:
    audio = make_synthetic_audio(emo, seed=9999 + CFG.EMOTIONS.index(emo))
    pred, probs = tta_predict(best_model, audio, processor, n_tta=5)
    ok = '✅' if pred == emo else '❌'
    print(f'{emo:<12}  {pred:<12}  {probs.max()*100:.1f}%      {ok}')

# Detailed visualisation for one emotion
demo_emo   = 'happy'
demo_audio = make_synthetic_audio(demo_emo, seed=12345)
pred, probs = tta_predict(best_model, demo_audio, processor, n_tta=5)
print(f'\n\nDetailed TTA result for: {demo_emo.upper()}')
display(ipd.Audio(demo_audio, rate=CFG.SR))
visualize_prediction(pred, probs, title=f'TTA Prediction (True: {demo_emo})')

## 🎤 Upload Audio File

In [ ]:
from google.colab import files
print('📁 Upload a .wav or .mp3 file:')
uploaded = files.upload()

for fname in uploaded:
     audio, sr_loaded = librosa.load(fname, sr=CFG.SR, duration=CFG.DURATION)
     print(f'File: {fname} | Duration: {len(audio)/CFG.SR:.2f}s | SR: {CFG.SR}')
     display(ipd.Audio(audio, rate=CFG.SR))

     pred, probs = tta_predict(best_model, audio, processor, n_tta=7)
     print(f'\n🎯 Predicted Emotion: {pred.upper()} (confidence: {probs.max()*100:.1f}%)')
     visualize_prediction(pred, probs, title=f'Your Audio → {pred.upper()}')

print('Uncomment the cell above to upload and analyse your own audio file.')

## 🧮 Model Ensemble

In [ ]:
@torch.no_grad()
def ensemble_predict(audio, processor, fold_models, n_tta=3):
    """
    Ensemble N fold-models × M TTA passes → average probabilities.
    Most robust inference strategy.
    """
    total_probs = np.zeros(CFG.N_CLASSES)
    for m in fold_models:
        _, probs = tta_predict(m, audio, processor, n_tta=n_tta)
        total_probs += probs
    avg_probs = total_probs / len(fold_models)
    return CFG.EMOTIONS[np.argmax(avg_probs)], avg_probs

# Load all fold models
fold_models = []
for r in fold_results:
    m = SOTASpeechEmotionModel(CFG).to(DEVICE)
    m.load_state_dict(torch.load(f'ser_model_fold{r["fold"]}.pt', map_location=DEVICE))
    m.eval()
    fold_models.append(m)

print(f'✅ Loaded {len(fold_models)} fold models for ensemble\n')

# Evaluate ensemble on all test samples
print('Running ensemble evaluation...')
ens_preds, ens_true = [], []
for emo in tqdm(CFG.EMOTIONS, desc='Ensemble eval'):
    for i in range(15):  # 15 held-out per class
        seed = 50000 + i * 13 + CFG.EMOTIONS.index(emo)
        audio = make_synthetic_audio(emo, seed=seed)
        pred, _ = ensemble_predict(audio, processor, fold_models, n_tta=3)
        ens_preds.append(CFG.EMOTIONS.index(pred))
        ens_true.append(CFG.EMOTIONS.index(emo))

ens_acc = accuracy_score(ens_true, ens_preds)
ens_f1  = f1_score(ens_true, ens_preds, average='weighted')
print(f'\n🏆 Ensemble Results:')
print(f'   Accuracy:    {ens_acc*100:.2f}%')
print(f'   Weighted F1: {ens_f1:.4f}')
print(f'   (vs single best fold — Acc: {best_fold["acc"]*100:.2f}%, F1: {best_fold["f1"]:.4f})')